In [1]:
 
import numpy as np 
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import math
import pandas as pd 
# import matplotlib.pyplot as plt

# import seaborn as sns
import sklearn.metrics as metrics
# from pandas_ml import ConfusionMatrix

In [2]:
train = pd.read_csv('UNSW_NB15_training-set.csv')
test = pd.read_csv('UNSW_NB15_testing-set.csv')
combined_data = pd.concat([train, test]).drop(['id'],axis=1)

In [3]:
from sklearn.preprocessing import LabelEncoder,normalize
le1 = LabelEncoder()
le = LabelEncoder()

vector = combined_data['attack_cat']
print("attack cat:", set(list(vector))) # use print to make it print on single line 

combined_data['attack_cat'] = le1.fit_transform(vector)
combined_data['proto'] = le.fit_transform(combined_data['proto'])
combined_data['service'] = le.fit_transform(combined_data['service'])
combined_data['state'] = le.fit_transform(combined_data['state'])

vector = combined_data['attack_cat']

attack cat: {'Analysis', 'Reconnaissance', 'Worms', 'Generic', 'Normal', 'Fuzzers', 'Shellcode', 'Backdoor', 'DoS', 'Exploits'}


In [4]:
le1.inverse_transform([0,1,2,3,4,5,6,7,8,9])
combined_data.head(3)

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0.000011,119,0,5,2,0,496,0,90909.0902,254,...,1,2,0,0,0,1,2,0,6,0
1,0.000008,119,0,5,2,0,1762,0,125000.0003,254,...,1,2,0,0,0,1,2,0,6,0
2,0.000005,119,0,5,2,0,1068,0,200000.0051,254,...,1,3,0,0,0,1,3,0,6,0


In [5]:
## OMITTED: For statistical feature removal

lowSTD = list(combined_data.std().to_frame().nsmallest(6, columns=0).index)
# this is stupid. suppose a feature has a 1.0 (spearman or pearson) correlation, OR conditional probability, when not 0.... That a very useful feature  

lowCORR = list(combined_data.corr().abs().sort_values('attack_cat')['attack_cat'].nsmallest(3).index) # .where(lambda x: x < 0.005).dropna()
# This might be stupid. A Deep MLP (feed forward neural net) may see patterns

drop = set( lowCORR + lowSTD)
drop = {'ackdat', 'ct_ftp_cmd', 'djit', 'is_ftp_login', 'is_sm_ips_ports', 'response_body_len', 'sjit', 'synack', 'tcprtt'}
# print(f'Before {combined_data.shape}')
combined_data_reduced=combined_data.drop(drop,axis=1)
# print(f'After {combined_data.shape}')

In [6]:
data_x = combined_data_reduced.drop(['attack_cat','label'], axis=1) # droped label
data_y = combined_data_reduced.loc[:,['label']]
# del combined_data # free mem
X_train, X_test, y_train, y_test = train_test_split(data_x, data_y, test_size=.20, random_state=42) # TODO

y_train = y_train.values.flatten()
y_test = y_test.values.flatten()

In [7]:
data_x2 = combined_data_reduced.drop(['attack_cat'], axis=1) # droped label
data_y2 = combined_data_reduced.loc[:,['label']]

X_train2, X_test2, y_train2, y_test2 = train_test_split(data_x2, data_y2, test_size=0.2, random_state=42)


In [8]:
Y_train = data_y2.values.flatten()
dict = {}
for i in Y_train:
    dict.update({i:dict.get(i,0)+1})
dict

{0: 93000, 1: 164673}

In [9]:
X_train.shape
data_y2.max()

label    1
dtype: int64

In [10]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape) # test is larger... good 
print(y_test.shape)
# y_train.min()

(206138, 33)
(206138,)
(51535, 33)
(51535,)


In [11]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import Normalizer

In [12]:
X = X_train
T = X_test
Y = y_train
C = y_test

scaler = Normalizer().fit(X)
trainX = scaler.transform(X)

scaler = Normalizer().fit(T)
testT = scaler.transform(T)

traindata = np.array(trainX)
trainlabel = np.array(Y)

testdata = np.array(testT)
testlabel = np.array(C)
testlabel = testlabel.flatten()

In [13]:
expected = testlabel
np.savetxt("EXP.txt", expected) 

In [14]:
from sklearn.ensemble import AdaBoostClassifier
from imblearn.ensemble import EasyEnsembleClassifier
from imblearn.ensemble import BalancedBaggingClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.ensemble import RUSBoostClassifier

In [15]:
EE = EasyEnsembleClassifier(n_estimators=10, n_jobs=1,
             replacement=False, sampling_strategy='auto',verbose=0, warm_start=False,
                                     base_estimator=DecisionTreeClassifier())
EE.fit(traindata, trainlabel)
predicted4 = EE.predict(testdata)
np.savetxt("EE.txt", predicted4) 

In [16]:
BB = BalancedBaggingClassifier(n_estimators=10, random_state=42)
BB.fit(traindata, trainlabel)
predicted5 = BB.predict(testdata)
np.savetxt("BB.txt", predicted5) 

In [17]:
RUSB = RUSBoostClassifier(random_state=42)
RUSB.fit(X_train, y_train)
BB.fit(traindata, trainlabel)
predicted6 = RUSB.predict(testdata)
np.savetxt("RUSB.txt", predicted6) 

D:\Anaconda3\envs\py38a\lib\site-packages\sklearn\base.py:450: UserWarning: X does not have valid feature names, but RUSBoostClassifier was fitted with feature names
  warnings.warn(


In [18]:
AB = AdaBoostClassifier(n_estimators=5,random_state=42)
AB.fit(X_train, y_train)
AB.fit(traindata, trainlabel)
predicted7 = AB.predict(testdata)
np.savetxt("AB.txt", predicted7) 

In [22]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(data_x, data_y)

data_x3 = np.append(data_x, X_resampled, axis = 0)
data_y3 = np.append(data_y.values, y_resampled)

In [23]:
# from imblearn.combine import SMOTEENN
# smote_enn = SMOTEENN(random_state=42)
# X_resampled, y_resampled = smote_enn.fit_resample(data_x, data_y)

# data_x3 = np.append(data_x, X_resampled, axis = 0)
# data_y3 = np.append(data_y.values, y_resampled)

In [24]:

# from imblearn.combine import SMOTETomek
# smote_tomek = SMOTETomek(random_state=42)
# X_resampled2, y_resampled2 = smote_tomek.fit_resample(data_x, data_y)

# data_x3 = np.append(data_x, X_resampled2, axis = 42)
# data_y3 = np.append(data_y.values, y_resampled2)

In [25]:
X_train, X_test, y_train, y_test = train_test_split(data_x3, data_y3, test_size=.20, random_state=42) # TODO

# y_train = y_train.values.flatten()
# y_test = y_test.values.flatten()

In [26]:
Y_train = data_y.values.flatten()
dict = {}
for i in Y_train:
    dict.update({i:dict.get(i,0)+1})
dict

{0: 93000, 1: 164673}

In [27]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape) # test is larger... good 
print(y_test.shape)

(469615, 33)
(469615,)
(117404, 33)
(117404,)


In [28]:
X = X_train
T = X_test
Y = y_train
C = y_test

scaler = Normalizer().fit(X)
trainX = scaler.transform(X)

scaler = Normalizer().fit(T)
testT = scaler.transform(T)

traindata = np.array(trainX)
trainlabel = np.array(Y)

testdata = np.array(testT)
testlabel = np.array(C)
testlabel = testlabel.flatten()

In [29]:
KNN = KNeighborsClassifier()
KNN.fit(traindata, trainlabel)

DT = DecisionTreeClassifier()
DT.fit(traindata, trainlabel)

RF = RandomForestClassifier(n_estimators=100)
RF.fit(traindata, trainlabel)

RandomForestClassifier()

In [30]:
expected2 = testlabel
np.savetxt("EXP2.txt", expected2) 
predicted1 =  KNN.predict(testdata)
np.savetxt("KNN.txt", predicted1)
predicted2 =  DT.predict(testdata)
np.savetxt("DT.txt", predicted2) 
predicted3 = RF.predict(testdata)
np.savetxt("RF.txt", predicted3) 
